# Thai Election OCR Pipeline (Form สส.6/1)
## Super AI Engineer Season 6 - OCR Competition 2569

Extract structured voting data from scanned Thai election result documents using Gemini Vision API.

**Pipeline:**
1. Download data from Kaggle
2. Group multi-page documents
3. Load sample labels to understand output format
4. OCR with Gemini 2.5 Flash (structured output)
5. Post-process & match party names
6. Generate submission CSV

## Cell 1: Setup & Dependencies

In [ ]:
!pip install -q google-genai kaggle tqdm Pillow

In [ ]:
import os
import re
import json
import time
import base64
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher

import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from google import genai
from google.genai import types
from google.colab import userdata

# === Configuration ===
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
MODEL = 'gemini-2.0-flash'  # 15 RPM free tier (vs 10 RPM for 2.5-flash)
DATA_DIR = Path('/content/data')
IMAGE_DIR = DATA_DIR / 'images'
SAMPLE_LABELS_DIR = DATA_DIR / 'sample_labels'
SUBMISSION_TEMPLATE = DATA_DIR / 'submission_template.csv'
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_FILE = OUTPUT_DIR / 'ocr_results.json'

# Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

print(f'Setup complete. Model: {MODEL} (15 RPM free tier)')

## Cell 2: Download Data from Kaggle

In [ ]:
# === Kaggle Authentication & Download ===
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)

# ตั้ง credentials จาก Colab Secrets
from google.colab import userdata
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": username, "key": key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print(f'Credentials set: username={username}')

# ทดสอบ auth ก่อน
print('\n=== ทดสอบ Kaggle API ===')
!kaggle competitions list --search "super-ai-engineer" 2>&1

# ถ้าเห็น competition = auth ผ่าน
# ถ้าเห็น 401/403 = ยังมีปัญหา credentials
# ถ้าเห็น list เปล่า = auth ผ่านแต่ไม่เจอ competition

In [ ]:
COMPETITION = 'super-ai-engineer-season-6-ocr-2569'

# ลองโหลด — ถ้า 401 = ยังไม่ได้ Accept Rules บนเว็บ Kaggle
# ไปที่: https://www.kaggle.com/competitions/super-ai-engineer-season-6-ocr-2569/rules
# เลื่อนลงล่างสุด กด "I Understand and Accept"
!kaggle competitions download -c {COMPETITION} -p /content/ 2>&1

# ถ้า Kaggle API ไม่ผ่านจริงๆ ให้ใช้ Manual Upload แทน:
# 1. เปิด https://www.kaggle.com/competitions/super-ai-engineer-season-6-ocr-2569/data
# 2. กด "Download All"
# 3. Uncomment 4 บรรทัดข้างล่างแล้วรัน

# from google.colab import files
# print('Upload zip file จาก Kaggle:')
# uploaded = files.upload()
# print('Upload done!')

In [ ]:
import glob as glob_module

# Unzip ทุกไฟล์ zip
zip_files = glob_module.glob('/content/*.zip')
print(f'Found zip files: {zip_files}')
for zf in zip_files:
    print(f'Extracting: {zf}')
    !unzip -qo "{zf}" -d /content/

# Auto-discover data paths
found_images = glob_module.glob('/content/**/images', recursive=True)
if found_images:
    IMAGE_DIR = Path(found_images[0])
    DATA_DIR = IMAGE_DIR.parent
    SAMPLE_LABELS_DIR = DATA_DIR / 'sample_labels'
    SUBMISSION_TEMPLATE = DATA_DIR / 'submission_template.csv'
    print(f'DATA_DIR = {DATA_DIR}')

if not SUBMISSION_TEMPLATE.exists():
    found_tmpl = glob_module.glob('/content/**/submission_template.csv', recursive=True)
    if found_tmpl:
        SUBMISSION_TEMPLATE = Path(found_tmpl[0])

# === Verify ===
images = sorted(IMAGE_DIR.glob('*.png'))
print(f'\nTotal images: {len(images)}')
if images:
    print(f'First 5: {[img.name for img in images[:5]]}')
print(f'Sample labels: {SAMPLE_LABELS_DIR.exists()}')
print(f'Submission template: {SUBMISSION_TEMPLATE} exists={SUBMISSION_TEMPLATE.exists()}')

## Cell 3: Explore Sample Labels & Submission Template

In [ ]:
# Inspect sample labels to understand expected output format
if SAMPLE_LABELS_DIR.exists():
    for jf in sorted(SAMPLE_LABELS_DIR.glob('*.json'))[:3]:
        print(f'\n=== {jf.name} ===')
        with open(jf) as f:
            data = json.load(f)
        print(json.dumps(data, ensure_ascii=False, indent=2)[:2000])

In [ ]:
# Inspect submission template
template_df = pd.read_csv(SUBMISSION_TEMPLATE)
print(f'Template shape: {template_df.shape}')
print(f'Columns: {list(template_df.columns)}')
print(template_df.head(20))
print(f'\nUnique doc_ids: {template_df["doc_id"].nunique()}')
print(f'\nSample doc_ids:')
print(template_df['doc_id'].unique()[:10])

## Cell 4: Group Multi-Page Documents

In [ ]:
def group_documents(image_dir: Path) -> dict:
    """Group PNG files into documents. Returns {doc_id: [(page_num, path), ...]}."""
    docs = defaultdict(list)
    pattern = re.compile(r'^(.+?)(?:_page(\d+))?\.png$')

    for p in sorted(image_dir.glob('*.png')):
        m = pattern.match(p.name)
        if not m:
            print(f'WARNING: Unmatched filename: {p.name}')
            continue
        doc_id = m.group(1)
        page_num = int(m.group(2)) if m.group(2) else 1
        docs[doc_id].append((page_num, p))

    # Sort pages within each document
    for doc_id in docs:
        docs[doc_id].sort(key=lambda x: x[0])

    return dict(docs)

documents = group_documents(IMAGE_DIR)
print(f'Total documents: {len(documents)}')
print(f'Page counts: {dict(sorted(pd.Series([len(v) for v in documents.values()]).value_counts().items()))}')

# Show some examples
for doc_id in list(documents.keys())[:5]:
    pages = documents[doc_id]
    print(f'  {doc_id}: {len(pages)} pages -> {[p.name for _, p in pages]}')

In [ ]:
# Classify document types
def get_doc_type(doc_id: str) -> str:
    """Determine if document is constituency or partylist."""
    doc_lower = doc_id.lower()
    if 'partylist' in doc_lower or 'party_list' in doc_lower or 'บัญชีรายชื่อ' in doc_lower:
        return 'partylist'
    elif 'constituency' in doc_lower or 'แบ่งเขต' in doc_lower:
        return 'constituency'
    else:
        # Infer from template: partylist docs have ~57 rows, constituency 4-19
        if doc_id in template_doc_rows:
            return 'partylist' if template_doc_rows[doc_id] > 30 else 'constituency'
        return 'unknown'

# Build lookup of expected row counts per doc
template_doc_rows = template_df.groupby('doc_id').size().to_dict()

doc_types = {doc_id: get_doc_type(doc_id) for doc_id in documents}
type_counts = pd.Series(doc_types).value_counts()
print(f'Document types:\n{type_counts}')

## Cell 5: Build Expected Party Names per Document

In [ ]:
# Build mapping: doc_id -> [(row_num, party_name, submission_id)] from template
doc_expected = defaultdict(list)
for _, row in template_df.iterrows():
    doc_expected[row['doc_id']].append({
        'id': row['id'],
        'row_num': row['row_num'],
        'party_name': row['party_name'],
    })

# Sort by row_num
for doc_id in doc_expected:
    doc_expected[doc_id].sort(key=lambda x: x['row_num'])

print(f'Documents in template: {len(doc_expected)}')
print(f'Documents in images: {len(documents)}')

# Check for mismatches
missing_images = set(doc_expected.keys()) - set(documents.keys())
missing_template = set(documents.keys()) - set(doc_expected.keys())
if missing_images:
    print(f'WARNING: {len(missing_images)} docs in template but no images: {list(missing_images)[:5]}')
if missing_template:
    print(f'WARNING: {len(missing_template)} docs have images but not in template: {list(missing_template)[:5]}')

## Cell 6: Visualize Sample Documents

In [ ]:
import matplotlib.pyplot as plt

# Show first page of a few documents
sample_docs = list(documents.keys())[:4]
fig, axes = plt.subplots(1, len(sample_docs), figsize=(20, 8))
if len(sample_docs) == 1:
    axes = [axes]
for ax, doc_id in zip(axes, sample_docs):
    page1_path = documents[doc_id][0][1]
    img = Image.open(page1_path)
    ax.imshow(img)
    ax.set_title(f'{doc_id}\n({doc_types[doc_id]})', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Cell 7: Prompt Templates

In [ ]:
CONSTITUENCY_PROMPT = """คุณคือระบบ OCR ที่แม่นยำสำหรับเอกสารผลการเลือกตั้ง สส. ของไทย (แบบ สส.6/1 แบบแบ่งเขตเลือกตั้ง)
ภาพเหล่านี้คือหน้าต่างๆ จากเอกสารผลการเลือกตั้งอย่างเป็นทางการฉบับเดียวกัน

ให้ดึงข้อมูลจากตารางผลคะแนนทุกแถว โดยแต่ละแถวประกอบด้วย:
- หมายเลขประจำตัวผู้สมัคร
- ชื่อ-สกุล ผู้สมัคร
- สังกัดพรรคการเมือง
- คะแนนที่ได้รับ

กฎ:
- ดึงชื่อพรรคการเมืองตามที่ปรากฏในเอกสาร เป็นภาษาไทย
- ดึงคะแนนเป็นตัวเลขอารบิก (แปลงเลขไทย ๐-๙ เป็น 0-9)
- ลบเครื่องหมายจุลภาค (,) ออกจากตัวเลข
- ห้ามข้ามแถวใดๆ แม้คะแนนจะเป็น 0
- รวมข้อมูลจากทุกหน้าของเอกสารเดียวกัน
- ไม่ต้องสนใจหัวตาราง, ท้ายกระดาษ, ลายเซ็น, ตราประทับ
- ดึงเฉพาะแถวที่มีข้อมูลผู้สมัครจริงๆ เท่านั้น

ตอบเป็น JSON array เท่านั้น โดยแต่ละ element มี format:
{"party_name": "ชื่อพรรค", "votes": "คะแนน"}

ตัวอย่าง:
[{"party_name": "เพื่อไทย", "votes": "14813"}, {"party_name": "ก้าวไกล", "votes": "8921"}]

ตอบ JSON array เท่านั้น ห้ามมีข้อความอื่น"""

PARTYLIST_PROMPT = """คุณคือระบบ OCR ที่แม่นยำสำหรับเอกสารผลการเลือกตั้ง สส. ของไทย (แบบ สส.6/1 แบบบัญชีรายชื่อ)
ภาพเหล่านี้คือหน้าต่างๆ จากเอกสารผลการเลือกตั้งแบบบัญชีรายชื่ออย่างเป็นทางการฉบับเดียวกัน

ให้ดึงข้อมูลจากตารางผลคะแนนทุกแถว โดยแต่ละแถวประกอบด้วย:
- หมายเลขพรรค
- ชื่อพรรคการเมือง
- คะแนนที่ได้รับ

กฎ:
- ดึงชื่อพรรคการเมืองตามที่ปรากฏในเอกสาร เป็นภาษาไทย
- ดึงคะแนนเป็นตัวเลขอารบิก (แปลงเลขไทย ๐-๙ เป็น 0-9)
- ลบเครื่องหมายจุลภาค (,) ออกจากตัวเลข
- ควรมีพรรคทั้งหมดประมาณ 57 พรรค
- รวมข้อมูลจากทุกหน้าของเอกสารเดียวกัน ห้ามมีพรรคซ้ำ
- ห้ามข้ามแถวใดๆ แม้คะแนนจะเป็น 0
- ไม่ต้องสนใจหัวตาราง, ท้ายกระดาษ, ลายเซ็น, ตราประทับ, แถวรวมคะแนน

ตอบเป็น JSON array เท่านั้น โดยแต่ละ element มี format:
{"party_name": "ชื่อพรรค", "votes": "คะแนน"}

ตัวอย่าง:
[{"party_name": "เพื่อไทย", "votes": "14813"}, {"party_name": "ก้าวไกล", "votes": "8921"}]

ตอบ JSON array เท่านั้น ห้ามมีข้อความอื่น"""

print('Prompts defined.')

## Cell 8: OCR Extraction Engine

In [ ]:
def load_checkpoint() -> dict:
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {}

def save_checkpoint(results: dict):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


def extract_document(doc_id: str, page_paths: list, doc_type: str) -> list:
    """Send all pages to Gemini, return list of {party_name, votes}."""
    prompt = CONSTITUENCY_PROMPT if doc_type == 'constituency' else PARTYLIST_PROMPT

    parts = []
    for _, page_path in page_paths:
        img = Image.open(page_path)
        parts.append(img)
    parts.append(prompt)

    response = client.models.generate_content(
        model=MODEL,
        contents=parts,
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=16384,
            response_mime_type="application/json",
        ),
    )

    text = response.text.strip()
    rows = json.loads(text)
    return rows


def save_partial_csv(results, template_df, doc_expected, output_dir):
    """สร้าง submission CSV จากผลที่มีอยู่ตอนนี้"""
    submission_votes = {}
    for doc_id, expected_rows in doc_expected.items():
        ocr_rows = results.get(doc_id, [])
        if not ocr_rows:
            for exp in expected_rows:
                submission_votes[exp['id']] = '0'
            continue
        # Simple positional match
        if len(ocr_rows) == len(expected_rows):
            for ocr_row, exp in zip(ocr_rows, expected_rows):
                votes = clean_votes(ocr_row.get('votes', '0'))
                submission_votes[exp['id']] = votes
        else:
            # Fallback: match what we can by position, fill rest with 0
            for i, exp in enumerate(expected_rows):
                if i < len(ocr_rows):
                    votes = clean_votes(ocr_rows[i].get('votes', '0'))
                    submission_votes[exp['id']] = votes
                else:
                    submission_votes[exp['id']] = '0'

    sub = template_df[['id']].copy()
    sub['votes'] = sub['id'].map(submission_votes).fillna('0')
    csv_path = output_dir / 'submission.csv'
    sub.to_csv(csv_path, index=False)
    return csv_path


print('Extraction engine ready.')

## Cell 9: Run OCR on All Documents

In [ ]:
# === Sequential extraction — Gemini 2.0 Flash (15 RPM) ===
results = load_checkpoint()
print(f'Loaded {len(results)} existing results from checkpoint.')

doc_ids_sorted = sorted(documents.keys(), key=lambda d: (doc_types.get(d, 'z'), d))
todo = [d for d in doc_ids_sorted if d not in results]
print(f'Documents to process: {len(todo)} (skipped {len(results)} already done)')

SAVE_EVERY = 5
DELAY = 4  # 15 RPM = 4s between requests (safe margin)
errors = []
start_time = time.time()

for i, doc_id in enumerate(tqdm(todo, desc='OCR Extraction')):
    doc_type = doc_types.get(doc_id, 'constituency')
    page_paths = documents[doc_id]

    for attempt in range(3):
        try:
            rows = extract_document(doc_id, page_paths, doc_type)
            results[doc_id] = rows

            expected_count = template_doc_rows.get(doc_id, -1)
            if expected_count > 0 and len(rows) != expected_count:
                print(f'  WARN {doc_id}: got {len(rows)} rows, expected {expected_count}')

            break  # success

        except json.JSONDecodeError as e:
            print(f'  JSON error {doc_id} (attempt {attempt+1}): {e}')
            time.sleep(10)
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'RESOURCE_EXHAUSTED' in err_str:
                wait = 60 * (attempt + 1)
                print(f'  Rate limited, waiting {wait}s...')
                time.sleep(wait)
            else:
                print(f'  Error {doc_id} (attempt {attempt+1}): {e}')
                time.sleep(10)
    else:
        errors.append(doc_id)
        print(f'  FAILED: {doc_id}')

    # Save checkpoint + CSV periodically
    done_count = len(results)
    if done_count % SAVE_EVERY == 0 or i == len(todo) - 1:
        save_checkpoint(results)
        csv_path = save_partial_csv(results, template_df, doc_expected, OUTPUT_DIR)
        elapsed = time.time() - start_time
        remaining = (elapsed / max(i+1, 1)) * (len(todo) - i - 1)
        print(f'  [{done_count}/{len(documents)}] {elapsed/60:.1f}min elapsed, ~{remaining/60:.0f}min left')

    time.sleep(DELAY)

# Final save
save_checkpoint(results)
save_partial_csv(results, template_df, doc_expected, OUTPUT_DIR)

elapsed = time.time() - start_time
print(f'\nDone! {len(results)}/{len(documents)} docs in {elapsed/60:.1f} min')
if errors:
    print(f'Failed ({len(errors)}): {errors}')
print(f'Submission: {OUTPUT_DIR}/submission.csv')

## Cell 10: Retry Failed Documents (if any)

In [ ]:
# Retry any documents that failed or are missing
results = load_checkpoint()
missing = [d for d in documents if d not in results]
print(f'Missing documents: {len(missing)}')

for doc_id in tqdm(missing, desc='Retry'):
    doc_type = doc_types.get(doc_id, 'constituency')
    page_paths = documents[doc_id]
    try:
        rows = extract_document(doc_id, page_paths, doc_type)
        results[doc_id] = rows
        save_checkpoint(results)
        time.sleep(RPM_DELAY)
    except Exception as e:
        print(f'  Still failing: {doc_id}: {e}')
        time.sleep(10)

print(f'Total results: {len(results)}/{len(documents)}')

## Cell 11: Post-Processing

In [ ]:
# === Thai numeral conversion ===
THAI_TO_ARABIC = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')

def clean_votes(raw) -> str:
    """Convert raw vote string to clean Arabic numeral string."""
    if isinstance(raw, (int, float)):
        return str(int(raw))
    s = str(raw).translate(THAI_TO_ARABIC)
    s = re.sub(r'[^0-9]', '', s)  # Remove commas, spaces, dots, etc.
    return s if s else '0'


def normalize_party_name(name: str) -> str:
    """Normalize party name for matching (remove extra whitespace)."""
    return re.sub(r'\s+', ' ', name.strip())


def fuzzy_match_party(ocr_name: str, candidates: list, threshold=0.55) -> tuple:
    """
    Match OCR party name to candidate list using fuzzy matching.
    Returns (best_match, score).
    """
    ocr_norm = normalize_party_name(ocr_name)

    # Exact match first
    for c in candidates:
        if normalize_party_name(c) == ocr_norm:
            return c, 1.0

    # Substring containment
    for c in candidates:
        c_norm = normalize_party_name(c)
        if ocr_norm in c_norm or c_norm in ocr_norm:
            return c, 0.95

    # Fuzzy match
    best, best_score = None, 0
    for c in candidates:
        score = SequenceMatcher(None, ocr_norm, normalize_party_name(c)).ratio()
        if score > best_score:
            best, best_score = c, score

    if best_score >= threshold:
        return best, best_score
    return None, best_score


print('Post-processing functions defined.')

## Cell 12: Match OCR Results to Submission Template

In [ ]:
results = load_checkpoint()

# Build submission mapping: id -> votes
submission_votes = {}  # submission_id -> votes string
match_issues = []  # Track problematic matches

for doc_id, expected_rows in tqdm(doc_expected.items(), desc='Matching'):
    ocr_rows = results.get(doc_id, [])

    if not ocr_rows:
        # No OCR result — fill with '0'
        for exp in expected_rows:
            submission_votes[exp['id']] = '0'
        match_issues.append({'doc_id': doc_id, 'issue': 'no_ocr_result'})
        continue

    # Get expected party names for this document
    expected_parties = [exp['party_name'] for exp in expected_rows]

    # Strategy 1: Try to match by position (row order) if counts match
    if len(ocr_rows) == len(expected_rows):
        # Verify that party names roughly match by position
        position_match_score = 0
        for ocr_row, exp in zip(ocr_rows, expected_rows):
            _, score = fuzzy_match_party(ocr_row['party_name'], [exp['party_name']])
            position_match_score += score
        avg_score = position_match_score / len(ocr_rows)

        if avg_score > 0.5:  # Good positional match
            for ocr_row, exp in zip(ocr_rows, expected_rows):
                votes = clean_votes(ocr_row.get('votes', '0'))
                submission_votes[exp['id']] = votes
            continue

    # Strategy 2: Match by party name (fuzzy matching)
    remaining_expected = list(range(len(expected_rows)))
    matched_ocr = set()

    # First pass: exact/high-confidence matches
    for i, ocr_row in enumerate(ocr_rows):
        ocr_party = ocr_row.get('party_name', '')
        remaining_party_names = [expected_rows[j]['party_name'] for j in remaining_expected]
        match, score = fuzzy_match_party(ocr_party, remaining_party_names, threshold=0.6)

        if match:
            # Find the index in expected_rows
            for j in remaining_expected:
                if expected_rows[j]['party_name'] == match:
                    votes = clean_votes(ocr_row.get('votes', '0'))
                    submission_votes[expected_rows[j]['id']] = votes
                    remaining_expected.remove(j)
                    matched_ocr.add(i)
                    break
        else:
            if score < 0.4:
                match_issues.append({
                    'doc_id': doc_id,
                    'issue': 'low_match',
                    'ocr_party': ocr_party,
                    'best_score': round(score, 3),
                })

    # Fill unmatched expected rows with '0'
    for j in remaining_expected:
        if expected_rows[j]['id'] not in submission_votes:
            submission_votes[expected_rows[j]['id']] = '0'
            match_issues.append({
                'doc_id': doc_id,
                'issue': 'unmatched_expected',
                'party_name': expected_rows[j]['party_name'],
            })

print(f'\nMatched {len(submission_votes)} / {len(template_df)} submission rows.')
print(f'Issues: {len(match_issues)}')
if match_issues:
    issue_df = pd.DataFrame(match_issues)
    print(issue_df['issue'].value_counts())

## Cell 13: Investigate Match Issues (Optional)

In [ ]:
if match_issues:
    issue_df = pd.DataFrame(match_issues)
    print('=== Low Match Issues ===')
    low_matches = issue_df[issue_df['issue'] == 'low_match']
    if len(low_matches) > 0:
        print(low_matches.head(20).to_string())

    print('\n=== Unmatched Expected ===')
    unmatched = issue_df[issue_df['issue'] == 'unmatched_expected']
    if len(unmatched) > 0:
        print(f'{len(unmatched)} unmatched rows')
        print(unmatched.head(20).to_string())

    print('\n=== No OCR Result ===')
    no_ocr = issue_df[issue_df['issue'] == 'no_ocr_result']
    if len(no_ocr) > 0:
        print(f'{len(no_ocr)} documents with no OCR result')

## Cell 14: Generate Submission CSV

In [ ]:
# สร้าง submission จาก submission_template.csv
# Kaggle ต้องการแค่ 2 คอลัมน์: id, votes
submission = template_df[['id']].copy()
submission['votes'] = submission['id'].map(submission_votes).fillna('0')

# Validate
filled = submission[submission['votes'] != '0']
print(f'Submission shape: {submission.shape}')
print(f'Rows with votes: {len(filled)} / {len(submission)}')
print(f'Rows with 0: {len(submission) - len(filled)}')

# Save
output_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(output_path, index=False)
print(f'\nSaved to: {output_path}')
print(submission.head(20))

## Cell 15: Validate Against Sample Labels

In [ ]:
def levenshtein_distance(s1: str, s2: str) -> int:
    """Compute Levenshtein distance between two strings."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            cost = 0 if c1 == c2 else 1
            curr_row.append(min(
                curr_row[j] + 1,       # insertion
                prev_row[j + 1] + 1,   # deletion
                prev_row[j] + cost,     # substitution
            ))
        prev_row = curr_row
    return prev_row[-1]


# Validate against sample labels if available
if SAMPLE_LABELS_DIR.exists():
    total_dist = 0
    total_pairs = 0

    for jf in sorted(SAMPLE_LABELS_DIR.glob('*.json')):
        with open(jf) as f:
            label_data = json.load(f)

        doc_id = jf.stem
        print(f'\n=== {doc_id} ===')

        # Try to match label format — adapt based on actual structure
        if isinstance(label_data, list):
            label_rows = label_data
        elif isinstance(label_data, dict) and 'rows' in label_data:
            label_rows = label_data['rows']
        else:
            print(f'  Unknown label format: {type(label_data)}')
            continue

        # Compare with our OCR results
        ocr_rows = results.get(doc_id, [])
        print(f'  Label rows: {len(label_rows)}, OCR rows: {len(ocr_rows)}')

        for label_row in label_rows:
            actual_votes = str(label_row.get('votes', label_row.get('คะแนน', '0')))
            party = label_row.get('party_name', label_row.get('ชื่อพรรค', ''))

            # Find matching OCR row
            predicted_votes = '0'
            for ocr_row in ocr_rows:
                match, score = fuzzy_match_party(
                    ocr_row.get('party_name', ''), [party]
                )
                if match and score > 0.5:
                    predicted_votes = clean_votes(ocr_row.get('votes', '0'))
                    break

            dist = levenshtein_distance(actual_votes, predicted_votes)
            total_dist += dist
            total_pairs += 1

            if dist > 0:
                print(f'  {party}: actual={actual_votes}, predicted={predicted_votes}, dist={dist}')

    if total_pairs > 0:
        mean_dist = total_dist / total_pairs
        print(f'\n=== Sample Validation ===')
        print(f'Total pairs: {total_pairs}')
        print(f'Total distance: {total_dist}')
        print(f'Mean Levenshtein distance: {mean_dist:.4f}')
else:
    print('No sample labels found.')

## Cell 16: Download Submission

In [ ]:
from google.colab import files
files.download(str(OUTPUT_DIR / 'submission.csv'))

## Cell 17: Re-process Specific Documents (Manual Fix)

Use this cell to re-process specific documents that had issues.

In [ ]:
# List documents to re-process
REPROCESS_DOCS = []  # Add doc_ids here, e.g., ['constituency_10_1', 'partylist_20_3']

# Optionally use a more powerful model for retries
RETRY_MODEL = 'gemini-2.5-flash'  # or 'gemini-2.5-pro' for harder cases

if REPROCESS_DOCS:
    for doc_id in tqdm(REPROCESS_DOCS, desc='Re-processing'):
        if doc_id not in documents:
            print(f'  Doc not found: {doc_id}')
            continue
        doc_type = doc_types.get(doc_id, 'constituency')
        page_paths = documents[doc_id]

        old_model = MODEL
        try:
            # Temporarily use retry model
            rows = extract_document(doc_id, page_paths, doc_type)
            results[doc_id] = rows
            print(f'  {doc_id}: {len(rows)} rows extracted')
            time.sleep(RPM_DELAY)
        except Exception as e:
            print(f'  Error: {doc_id}: {e}')

    save_checkpoint(results)
    print('Re-processing done. Re-run Cell 12+ to rebuild submission.')
else:
    print('No documents to re-process. Add doc_ids to REPROCESS_DOCS list above.')